In [ ]:
import shutil
INPUT_PROJECT_DIR = "/kaggle/input/notebooks/tanveerheir/notebook6d954a2374/dpr_rag_project"

if os.path.exists(INPUT_PROJECT_DIR):
    print("Found previous artifacts — copying to working directory...")
    shutil.copytree(INPUT_PROJECT_DIR, WORK_DIR, dirs_exist_ok=True)
    print("Done.")
else:
    print("Path not found — double check the path.")

# Verify what we now have
for root, dirs, files in os.walk(WORK_DIR):
    for f in files:
        full = os.path.join(root, f)
        print(full, f"({os.path.getsize(full)/1e6:.1f} MB)")

In [ ]:
# Cell A: Installs + imports (run this fresh every new session)
!pip install -q faiss-cpu transformers datasets sentence-transformers rank-bm25 sacrebleu accelerate evaluate

import os, json, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel,
    T5Tokenizer, T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from datasets import load_dataset
import faiss
from rank_bm25 import BM25Okapi


SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

WORK_DIR = "/kaggle/working/dpr_rag_project"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{WORK_DIR}/indexes", exist_ok=True)
os.makedirs(f"{WORK_DIR}/results", exist_ok=True)
print("Work dir ready:", WORK_DIR)

In [ ]:
# Cell B: Reload passages, train/val splits — rebuild from HF only if missing

passages_path = f"{WORK_DIR}/passages.json"
train_path = f"{WORK_DIR}/train_pairs.json"
val_path = f"{WORK_DIR}/val_pairs.json"

if os.path.exists(passages_path) and os.path.exists(train_path) and os.path.exists(val_path):
    print("Found existing data splits — loading from disk.")
    with open(passages_path) as f:
        passages = json.load(f)
    with open(train_path) as f:
        train_pairs = json.load(f)
    with open(val_path) as f:
        val_pairs = json.load(f)
else:
    print("No existing splits found — rebuilding from SQuAD.")
    raw = load_dataset("rajpurkar/squad", split="train")
    N_EXAMPLES = 25000
    raw = raw.shuffle(seed=SEED).select(range(min(N_EXAMPLES, len(raw))))

    seen = {}
    passages = []
    for ex in raw:
        ctx = ex["context"]
        if ctx not in seen:
            pid = len(passages)
            seen[ctx] = pid
            passages.append({"id": pid, "text": ctx, "title": ex["title"]})

    qa_pairs = []
    for ex in raw:
        pid = seen[ex["context"]]
        qa_pairs.append({"question": ex["question"], "pos_id": pid, "answers": ex["answers"]["text"]})

    random.shuffle(qa_pairs)
    n_val = 1000
    val_pairs = qa_pairs[:n_val]
    train_pairs = qa_pairs[n_val:]

    with open(passages_path, "w") as f: json.dump(passages, f)
    with open(train_path, "w") as f: json.dump(train_pairs, f)
    with open(val_path, "w") as f: json.dump(val_pairs, f)

print("Passages:", len(passages), "| Train pairs:", len(train_pairs), "| Val pairs:", len(val_pairs))

In [ ]:
# Cell C: Model definitions, dataset/loader, loss function (always rebuild — cheap)

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class Encoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]

question_encoder = Encoder().to(device)
passage_encoder = Encoder().to(device)

class DPRDataset(Dataset):
    def __init__(self, qa_pairs, passages):
        self.qa_pairs = qa_pairs
        self.passages = passages
    def __len__(self):
        return len(self.qa_pairs)
    def __getitem__(self, idx):
        item = self.qa_pairs[idx]
        return item["question"], self.passages[item["pos_id"]]["text"]

def collate_fn(batch):
    questions, pos_texts = zip(*batch)
    return list(questions), list(pos_texts)

BATCH_SIZE = 8
MAX_LEN_Q = 48
MAX_LEN_P = 160

train_dataset = DPRDataset(train_pairs, passages)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=collate_fn, drop_last=True)

def encode_texts(encoder, texts, max_len=160):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        emb = encoder(enc["input_ids"], enc["attention_mask"])
    return emb

def in_batch_negative_loss(q_emb, p_emb, temperature=0.1):
    q_emb = F.normalize(q_emb, p=2, dim=1)
    p_emb = F.normalize(p_emb, p=2, dim=1)
    scores = torch.matmul(q_emb, p_emb.T) / temperature
    labels = torch.arange(scores.size(0), device=scores.device)
    loss = F.cross_entropy(scores, labels)
    return loss, scores

EPOCHS = 3
LR = 2e-5
TEMP = 0.1
params = list(question_encoder.parameters()) + list(passage_encoder.parameters())
optimizer = torch.optim.AdamW(params, lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)

def train_step(questions, pos_texts):
    question_encoder.train()
    passage_encoder.train()
    q_enc = tokenizer(questions, padding=True, truncation=True, max_length=MAX_LEN_Q, return_tensors="pt")
    p_enc = tokenizer(pos_texts, padding=True, truncation=True, max_length=MAX_LEN_P, return_tensors="pt")
    q_emb = question_encoder(q_enc["input_ids"].to(device), q_enc["attention_mask"].to(device))
    p_emb = passage_encoder(p_enc["input_ids"].to(device), p_enc["attention_mask"].to(device))
    loss, _ = in_batch_negative_loss(q_emb, p_emb, temperature=TEMP)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
    optimizer.step()
    scheduler.step()
    loss_val = loss.item()
    del q_emb, p_emb, loss
    return loss_val

print("Architecture, dataset, and loss function ready.")
print("Steps per epoch:", len(train_loader), "| Total steps:", total_steps)

In [ ]:
# Cell D: Train only if checkpoints don't already exist

ckpt_q_path = f"{WORK_DIR}/checkpoints/question_encoder.pt"
ckpt_p_path = f"{WORK_DIR}/checkpoints/passage_encoder.pt"

if os.path.exists(ckpt_q_path) and os.path.exists(ckpt_p_path):
    print("Found existing checkpoints — loading instead of retraining.")
    question_encoder.load_state_dict(torch.load(ckpt_q_path, map_location=device))
    passage_encoder.load_state_dict(torch.load(ckpt_p_path, map_location=device))
    question_encoder.to(device)
    passage_encoder.to(device)
    print("Loaded trained encoders from disk.")
else:
    print("No checkpoints found — training from scratch.")
    train_losses = []
    log_every = 50
    start = time.time()
    for epoch in range(EPOCHS):
        epoch_losses = []
        for step, (questions, pos_texts) in enumerate(train_loader):
            loss_val = train_step(questions, pos_texts)
            epoch_losses.append(loss_val)
            train_losses.append(loss_val)
            if step % log_every == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(train_loader)} | Loss: {loss_val:.4f}")
        print(f"==> Epoch {epoch+1} avg loss: {np.mean(epoch_losses):.4f}")
    elapsed = (time.time() - start) / 60
    print(f"\nTotal training time: {elapsed:.1f} min")
    torch.save(question_encoder.state_dict(), ckpt_q_path)
    torch.save(passage_encoder.state_dict(), ckpt_p_path)
    print("Saved encoder checkpoints to", f"{WORK_DIR}/checkpoints/")

In [ ]:
# Cell E: Encode passage corpus only if embeddings don't already exist

emb_path = f"{WORK_DIR}/indexes/passage_embeddings.npy"

def encode_corpus(encoder, passages, batch_size=32, max_len=160):
    encoder.eval()
    all_embeddings = []
    texts = [p["text"] for p in passages]
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            emb = encoder(enc["input_ids"], enc["attention_mask"])
            all_embeddings.append(emb.cpu().numpy())
    return np.vstack(all_embeddings)

if os.path.exists(emb_path):
    print("Found existing passage embeddings — loading from disk.")
    passage_embeddings = np.load(emb_path)
else:
    print("No embeddings found — encoding corpus now.")
    start = time.time()
    passage_embeddings = encode_corpus(passage_encoder, passages, batch_size=32)
    print(f"Done in {(time.time()-start):.1f}s")
    np.save(emb_path, passage_embeddings)
    print("Saved embeddings to disk")

print("Passage embeddings shape:", passage_embeddings.shape)

In [ ]:
# Cell F: Build FAISS index only if it doesn't already exist on disk

index_path = f"{WORK_DIR}/indexes/faiss_index.bin"

if os.path.exists(index_path):
    print("Found existing FAISS index — loading from disk.")
    index = faiss.read_index(index_path)
else:
    print("No FAISS index found — building now.")
    embedding_dim = passage_embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)
    passage_embeddings = passage_embeddings.astype("float32")
    index.add(passage_embeddings)
    faiss.write_index(index, index_path)
    print("Saved FAISS index to disk")

print("Number of vectors in index:", index.ntotal)

In [ ]:
# Cell G: Rebuild BM25 index — fast, so we always rebuild rather than persist it

def simple_tokenize(text):
    return text.lower().split()

tokenized_corpus = [simple_tokenize(p["text"]) for p in passages]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index rebuilt over", len(tokenized_corpus), "passages")

In [ ]:
# Cell H: Redefine retrieval functions — always rebuild, these are cheap function defs

def dense_retrieve(question, k=5):
    question_encoder.eval()
    with torch.no_grad():
        enc = tokenizer([question], padding=True, truncation=True, max_length=MAX_LEN_Q, return_tensors="pt")
        enc = {k_: v.to(device) for k_, v in enc.items()}
        q_emb = question_encoder(enc["input_ids"], enc["attention_mask"])
        q_emb = q_emb.cpu().numpy().astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"passage_id": int(idx), "text": passages[idx]["text"], "score": float(score)})
    return results

def normalize_scores(scores):
    scores = np.array(scores)
    if scores.max() == scores.min():
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())

def hybrid_retrieve(question, k=5, alpha=0.5, dense_candidates=100, bm25_candidates=100):
    dense_results = dense_retrieve(question, k=dense_candidates)
    dense_ids = [r["passage_id"] for r in dense_results]
    dense_scores_map = {r["passage_id"]: r["score"] for r in dense_results}

    bm25_scores_all = bm25.get_scores(simple_tokenize(question))
    bm25_top_idx = np.argsort(bm25_scores_all)[::-1][:bm25_candidates]
    bm25_scores_map = {int(idx): float(bm25_scores_all[idx]) for idx in bm25_top_idx}

    candidate_ids = list(set(dense_ids) | set(bm25_scores_map.keys()))
    dense_vals = [dense_scores_map.get(pid, 0.0) for pid in candidate_ids]
    bm25_vals = [bm25_scores_map.get(pid, 0.0) for pid in candidate_ids]

    dense_norm = normalize_scores(dense_vals)
    bm25_norm = normalize_scores(bm25_vals)
    combined = alpha * dense_norm + (1 - alpha) * bm25_norm

    ranked = sorted(zip(candidate_ids, combined), key=lambda x: x[1], reverse=True)[:k]
    return [{"passage_id": pid, "text": passages[pid]["text"], "score": float(score)} for pid, score in ranked]

print("Retrieval functions ready: dense_retrieve(), hybrid_retrieve()")

In [ ]:
# Cell I: Quick end-to-end sanity check after reload

test_q = val_pairs[0]["question"]
print("Test question:", test_q)
print("\nDense top-3:")
for r in dense_retrieve(test_q, k=3):
    print(f"  score={r['score']:.3f} | {r['text'][:80]}...")
print("\nHybrid top-3:")
for r in hybrid_retrieve(test_q, k=3):
    print(f"  score={r['score']:.3f} | {r['text'][:80]}...")
print("\nGold passage:", passages[val_pairs[0]['pos_id']]['text'][:80], "...")

In [ ]:
# Cell 18: Sweep alpha to find the best dense/BM25 blend weight

def evaluate_recall_at_k_hybrid(qa_pairs, k_values=[1, 5, 20, 100], alpha=0.5):
    max_k = max(k_values)
    hits = {k: 0 for k in k_values}
    for item in qa_pairs:
        question = item["question"]
        gold_pid = item["pos_id"]
        retrieved = hybrid_retrieve(question, k=max_k, alpha=alpha)
        retrieved_ids = [r["passage_id"] for r in retrieved]
        for k in k_values:
            if gold_pid in retrieved_ids[:k]:
                hits[k] += 1
    n = len(qa_pairs)
    return {k: hits[k] / n for k in k_values}

alphas_to_try = [0.0, 0.3, 0.5, 0.7, 1.0]  # 0.0 = pure BM25, 1.0 = pure dense
ablation_results = {}

for alpha in alphas_to_try:
    start = time.time()
    recall = evaluate_recall_at_k_hybrid(val_pairs, k_values=[1, 5, 20, 100], alpha=alpha)
    elapsed = time.time() - start
    ablation_results[alpha] = recall
    print(f"alpha={alpha:.1f} | Recall@1={recall[1]:.4f} | Recall@5={recall[5]:.4f} | "
          f"Recall@20={recall[20]:.4f} | Recall@100={recall[100]:.4f} | ({elapsed:.1f}s)")

with open(f"{WORK_DIR}/results/alpha_ablation.json", "w") as f:
    json.dump(ablation_results, f, indent=2)

print("\nSaved ablation results to disk")

In [ ]:
# Cell 19: Load T5-base for FiD-style answer generation

GEN_MODEL_NAME = "t5-base"

gen_tokenizer = T5Tokenizer.from_pretrained(GEN_MODEL_NAME)
generator = T5ForConditionalGeneration.from_pretrained(GEN_MODEL_NAME).to(device)

n_params = sum(p.numel() for p in generator.parameters())
print(f"T5-base loaded: ~{n_params/1e6:.1f}M parameters")
print("Generator device:", next(generator.parameters()).device)

In [ ]:
# Cell 20: Build the FiD-style input — concatenate question + retrieved passages

def build_fid_input(question, retrieved_passages):
    passage_texts = [p["text"] for p in retrieved_passages]
    context_block = " ".join([f"context: {p}" for p in passage_texts])
    input_text = f"question: {question} {context_block}"
    return input_text

# Quick test
test_question = val_pairs[0]["question"]
retrieved = dense_retrieve(test_question, k=3)
fid_input = build_fid_input(test_question, retrieved)

print("Question:", test_question)
print("\nConstructed FiD input (first 500 chars):")
print(fid_input[:500], "...")
print("\nTotal input length (chars):", len(fid_input))

In [ ]:
# Cell 21: Generate an answer given the FiD-style input

def generate_answer(question, k=3, max_new_tokens=32, retriever_fn=dense_retrieve):
    retrieved = retriever_fn(question, k=k)
    fid_input = build_fid_input(question, retrieved)

    inputs = gen_tokenizer(fid_input, return_tensors="pt", truncation=True, max_length=512).to(device)

    generator.eval()
    with torch.no_grad():
        output_ids = generator.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer, retrieved

# Quick test
test_question = val_pairs[0]["question"]
answer, retrieved = generate_answer(test_question, k=3)

print("Question:", test_question)
print("Generated answer:", answer)
print("Gold answer(s):", val_pairs[0]["answers"])
print("\nTop retrieved passage used:")
print(retrieved[0]["text"][:200], "...")

In [ ]:
# Cell 22: Generate answers for a few examples, comparing dense-only vs hybrid retrieval

N_EXAMPLES_TO_SHOW = 5

print(f"{'='*80}\n")
for i in range(N_EXAMPLES_TO_SHOW):
    item = val_pairs[i]
    question = item["question"]
    gold_answers = item["answers"]

    dense_answer, _ = generate_answer(question, k=3, retriever_fn=dense_retrieve)
    hybrid_answer, _ = generate_answer(question, k=3, retriever_fn=lambda q, k: hybrid_retrieve(q, k=k, alpha=0.5))

    print(f"Q: {question}")
    print(f"Gold answer(s): {gold_answers}")
    print(f"Dense-retrieval generated:  {dense_answer}")
    print(f"Hybrid-retrieval generated: {hybrid_answer}")
    print(f"\n{'='*80}\n")

In [ ]:
# Cell 23: Exact Match (EM) and F1 evaluation, comparing dense vs hybrid retrieval-based generation

import re
import string
from collections import Counter

def normalize_answer(s):
    """Standard SQuAD-style normalization: lowercase, remove punctuation/articles/extra whitespace."""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_em(prediction, gold_answers):
    pred_norm = normalize_answer(prediction)
    return max(int(pred_norm == normalize_answer(g)) for g in gold_answers)

def compute_f1(prediction, gold_answers):
    pred_tokens = normalize_answer(prediction).split()
    best_f1 = 0.0
    for gold in gold_answers:
        gold_tokens = normalize_answer(gold).split()
        common = Counter(pred_tokens) & Counter(gold_tokens)
        num_same = sum(common.values())
        if len(pred_tokens) == 0 or len(gold_tokens) == 0:
            f1 = int(pred_tokens == gold_tokens)
        elif num_same == 0:
            f1 = 0.0
        else:
            precision = num_same / len(pred_tokens)
            recall = num_same / len(gold_tokens)
            f1 = 2 * precision * recall / (precision + recall)
        best_f1 = max(best_f1, f1)
    return best_f1

def evaluate_generation(qa_pairs, retriever_fn, k=3, n_samples=200):
    """Evaluate on a subset (n_samples) for speed — generation is slower than retrieval."""
    subset = qa_pairs[:n_samples]
    em_scores, f1_scores = [], []

    for item in subset:
        question = item["question"]
        gold_answers = item["answers"]
        prediction, _ = generate_answer(question, k=k, retriever_fn=retriever_fn)

        em_scores.append(compute_em(prediction, gold_answers))
        f1_scores.append(compute_f1(prediction, gold_answers))

    return {
        "EM": np.mean(em_scores),
        "F1": np.mean(f1_scores),
        "n_samples": len(subset)
    }

N_EVAL_SAMPLES = 200  # keep modest for generation speed; bump later if you want tighter numbers

print("Evaluating dense-retrieval-based generation...")
start = time.time()
dense_gen_results = evaluate_generation(val_pairs, retriever_fn=dense_retrieve, k=3, n_samples=N_EVAL_SAMPLES)
print(f"Done in {(time.time()-start)/60:.1f} min")
print(dense_gen_results)

print("\nEvaluating hybrid-retrieval-based generation...")
start = time.time()
hybrid_gen_results = evaluate_generation(
    val_pairs,
    retriever_fn=lambda q, k: hybrid_retrieve(q, k=k, alpha=0.5),
    k=3, n_samples=N_EVAL_SAMPLES
)
print(f"Done in {(time.time()-start)/60:.1f} min")
print(hybrid_gen_results)

print("\n=== Summary: Dense vs Hybrid Generation Quality ===")
print(f"{'Metric':<10}{'Dense':<10}{'Hybrid':<10}")
print(f"{'EM':<10}{dense_gen_results['EM']:<10.4f}{hybrid_gen_results['EM']:<10.4f}")
print(f"{'F1':<10}{dense_gen_results['F1']:<10.4f}{hybrid_gen_results['F1']:<10.4f}")

with open(f"{WORK_DIR}/results/generation_eval.json", "w") as f:
    json.dump({"dense": dense_gen_results, "hybrid": hybrid_gen_results}, f, indent=2)
print("\nSaved generation eval results to disk")

In [ ]:
# Cell 24: Single callable function — question in, full RAG answer out

def rag_pipeline(question, retrieval_mode="hybrid", k=3, alpha=0.5, max_new_tokens=32, verbose=True):
    """
    One function that ties the whole project together:
    question -> retrieve passages -> build FiD input -> generate answer

    retrieval_mode: "dense", "bm25", or "hybrid"
    """
    if retrieval_mode == "dense":
        retrieved = dense_retrieve(question, k=k)
    elif retrieval_mode == "bm25":
        retrieved = hybrid_retrieve(question, k=k, alpha=0.0)  # alpha=0 -> pure BM25
    elif retrieval_mode == "hybrid":
        retrieved = hybrid_retrieve(question, k=k, alpha=alpha)
    else:
        raise ValueError("retrieval_mode must be 'dense', 'bm25', or 'hybrid'")

    fid_input = build_fid_input(question, retrieved)
    inputs = gen_tokenizer(fid_input, return_tensors="pt", truncation=True, max_length=512).to(device)

    generator.eval()
    with torch.no_grad():
        output_ids = generator.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )
    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)

    if verbose:
        print(f"Question: {question}")
        print(f"Retrieval mode: {retrieval_mode} (k={k})")
        print(f"Top retrieved passage: {retrieved[0]['text'][:150]}...")
        print(f"Generated answer: {answer}")

    return {
        "question": question,
        "answer": answer,
        "retrieved_passages": retrieved,
        "retrieval_mode": retrieval_mode
    }

# Try it on a few of your own questions, not just SQuAD val examples
result = rag_pipeline("Which is ash ketchum first pokemon?", retrieval_mode="hybrid")
print()
result2 = rag_pipeline("What causes inflation in an economy?", retrieval_mode="hybrid")

In [ ]:
# Cell 25: Load all saved results and produce one consolidated summary

results_dir = f"{WORK_DIR}/results"

def safe_load(filename):
    path = f"{results_dir}/{filename}"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    else:
        print(f"Warning: {filename} not found — skipping.")
        return None

dense_recall = safe_load("dense_recall.json")
hybrid_recall = safe_load("hybrid_recall.json")
alpha_ablation = safe_load("alpha_ablation.json")
generation_eval = safe_load("generation_eval.json")

print("="*70)
print("MAJOR 1 — DENSE RETRIEVAL + RAG: FINAL RESULTS SUMMARY")
print("="*70)

print(f"\nCorpus: {len(passages)} unique passages (SQuAD subset, ~25K QA pairs)")
print(f"Validation set: {len(val_pairs)} questions")
print(f"Retriever: bert-base-uncased dual encoder, trained with in-batch negatives")
print(f"Generator: t5-base, FiD-style (simplified single-sequence variant)")

if dense_recall and hybrid_recall:
    print("\n--- Retrieval: Dense vs Hybrid (BM25 + Dense, alpha=0.5) Recall@k ---")
    print(f"{'k':<8}{'Dense':<12}{'Hybrid':<12}{'Gain':<10}")
    for k in ["1", "5", "20", "100"]:
        d = dense_recall.get(k, dense_recall.get(int(k)))
        h = hybrid_recall.get(k, hybrid_recall.get(int(k)))
        if d is not None and h is not None:
            print(f"{k:<8}{d:<12.4f}{h:<12.4f}{(h-d):<+10.4f}")

if alpha_ablation:
    print("\n--- Alpha Ablation (Recall@5 across blend weights) ---")
    print(f"{'alpha':<10}{'Recall@1':<12}{'Recall@5':<12}{'Recall@20':<12}{'Recall@100':<12}")
    for alpha_str, recall in sorted(alpha_ablation.items(), key=lambda x: float(x[0])):
        r1 = recall.get("1", recall.get(1))
        r5 = recall.get("5", recall.get(5))
        r20 = recall.get("20", recall.get(20))
        r100 = recall.get("100", recall.get(100))
        print(f"{float(alpha_str):<10.1f}{r1:<12.4f}{r5:<12.4f}{r20:<12.4f}{r100:<12.4f}")

if generation_eval:
    print("\n--- Generation Quality: EM / F1 (Dense vs Hybrid retrieval feeding T5-base) ---")
    print(f"{'Metric':<10}{'Dense':<12}{'Hybrid':<12}")
    print(f"{'EM':<10}{generation_eval['dense']['EM']:<12.4f}{generation_eval['hybrid']['EM']:<12.4f}")
    print(f"{'F1':<10}{generation_eval['dense']['F1']:<12.4f}{generation_eval['hybrid']['F1']:<12.4f}")
    print(f"(evaluated on {generation_eval['dense']['n_samples']} validation samples)")

print("\n" + "="*70)
print("Artifacts saved under:", WORK_DIR)
print("="*70)

# Save this summary as a single combined results file too
final_summary = {
    "corpus_size": len(passages),
    "val_set_size": len(val_pairs),
    "dense_recall": dense_recall,
    "hybrid_recall": hybrid_recall,
    "alpha_ablation": alpha_ablation,
    "generation_eval": generation_eval
}
with open(f"{results_dir}/final_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)
print("\nSaved consolidated final_summary.json")

In [ ]:
# Quick fix: regenerate dense_recall.json and hybrid_recall.json from the ablation table
dense_recall = alpha_ablation["1.0"]   # pure dense
hybrid_recall = alpha_ablation["0.5"]  # the alpha=0.5 hybrid blend we reported earlier

with open(f"{WORK_DIR}/results/dense_recall.json", "w") as f:
    json.dump(dense_recall, f, indent=2)
with open(f"{WORK_DIR}/results/hybrid_recall.json", "w") as f:
    json.dump(hybrid_recall, f, indent=2)

print("Re-saved dense_recall.json and hybrid_recall.json")